# 03 — MCP Demo

Conecte a um servidor MCP via stdio e chame tools.

In [ ]:
import sys; sys.path.insert(0, '..')

## 1. Schemas de Tools (Pydantic)

In [ ]:
import json
from modules.mcp.schemas.tool_definitions import EchoInput, AddNumbersInput, to_json_schema
print('Echo schema:')
print(json.dumps(to_json_schema(EchoInput), indent=2))
print('\nAddNumbers schema:')
print(json.dumps(to_json_schema(AddNumbersInput), indent=2))

## 2. Conectar ao Servidor MCP via Subprocess

Execute o bloco abaixo — ele inicia o servidor MCP como subprocess e faz chamadas.

In [ ]:
import asyncio
import sys
from pathlib import Path
from mcp import ClientSession, StdioServerParameters
from modules.mcp.client.stdio_client import notebook_safe_stdio_client

SERVER = str(Path('../modules/mcp/server/simple_server.py'))

async def demo():
    params = StdioServerParameters(command=sys.executable, args=[SERVER])
    async with notebook_safe_stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            print('Tools disponíveis:')
            for t in tools.tools:
                print(f'  • {t.name}: {t.description}')
            r1 = await session.call_tool('echo', {'text': 'Olá do notebook!'})
            print(f'\necho: {r1.content[0].text}')
            r2 = await session.call_tool('add_numbers', {'a': 42, 'b': 58})
            print(f'add_numbers(42, 58) = {r2.content[0].text}')

await demo()

## Exercícios

1. Adicione uma tool `multiply(a, b)` ao simple_server.py
2. Conecte ao file_server.py e liste os resources disponíveis
3. Implemente um servidor MCP que expõe o pipeline RAG como tool